# Data Preparation


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression

In [4]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

In [5]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [6]:
df.shape

(5000, 9)

In [7]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [8]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [9]:
categorical_columns = df.select_dtypes("str").columns


numerical_columns = df.select_dtypes (["float64", "int64"]).columns




In [10]:
df[categorical_columns] = df[categorical_columns].fillna("NA")
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [11]:
# verifying there are no nulls now
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# Q1: Find the mode of the industry

In [12]:
df.industry.mode()

0    technology
Name: industry, dtype: str

In [13]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

# Q2: Correlation Matrix

In [14]:
numerical_columns = df.select_dtypes(include="number").columns

df[numerical_columns].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [15]:
corr = df[numerical_columns].corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for feature_1, feature_2 in pairs:
    print(
        feature_1,
        "and",
        feature_2,
        ":",
        corr.loc[feature_1, feature_2]
    )

interaction_count and lead_score : 0.9157457980418697
number_of_courses_viewed and lead_score : 0.7572042392104886
number_of_courses_viewed and interaction_count : 0.7216090038937035
annual_income and interaction_count : 0.12284235135955959


# Splitting the data set

In [16]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [17]:
 df.shape , df_full_train.shape , df_train.shape, df_val.shape, df_test.shape

((5000, 9), (4000, 9), (3000, 9), (1000, 9), (1000, 9))

In [18]:
y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

In [19]:
del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

In [20]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(3000, 8)
(1000, 8)
(1000, 8)
(3000,)
(1000,)
(1000,)


# Question 3: Mutual information

In [21]:
from sklearn.metrics import mutual_info_score

In [22]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

In [23]:
categorical_columns = list(
    df_train.select_dtypes("str").columns
)

mi = df_train[categorical_columns].apply(
    mutual_info_converted_score
)

mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [24]:
# rounding for the accuracy of the answer.
mi[
    [
        "industry",
        "location",
        "lead_source",
        "employment_status"
    ]
].round(2).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64

# Question 4: Train a logistic regression model

In [25]:
from sklearn.feature_extraction import DictVectorizer

In [26]:
categorical_columns = list(
    df_train.select_dtypes(
        include=["str", "object", "category"]
    ).columns
)

numerical_columns = list(
    df_train.select_dtypes(include="number").columns
)

In [27]:
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

In [28]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [30]:
print(X_train.shape)
print(X_val.shape)

(3000, 28)
(1000, 28)


In [32]:
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_val.indices = X_val.indices.astype(np.int32)
X_val.indptr = X_val.indptr.astype(np.int32)

In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [34]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

print("Accuracy:", accuracy)
print("Rounded accuracy:", round(accuracy, 2))

Accuracy: 0.645
Rounded accuracy: 0.65


# Question 5: Feature elimination

In [35]:
original_accuracy = accuracy_score(
    y_val,
    model.predict(X_val)
)

print(original_accuracy)

0.645


In [36]:
features_to_check = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

for feature in features_to_check:
    # Remove one feature from the training and validation data
    train_dicts_reduced = df_train.drop(
        columns=[feature]
    ).to_dict(orient="records")

    val_dicts_reduced = df_val.drop(
        columns=[feature]
    ).to_dict(orient="records")

    # Encode the remaining features
    dv_reduced = DictVectorizer(sparse=True)

    X_train_reduced = dv_reduced.fit_transform(train_dicts_reduced)
    X_val_reduced = dv_reduced.transform(val_dicts_reduced)

    # Fix the sparse matrix index types for your environment
    for X in [X_train_reduced, X_val_reduced]:
        X.indices = X.indices.astype(np.int32)
        X.indptr = X.indptr.astype(np.int32)

    # Train a new model using the same parameters
    model_reduced = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_reduced.fit(X_train_reduced, y_train)

    # Evaluate accuracy
    reduced_accuracy = accuracy_score(
        y_val,
        model_reduced.predict(X_val_reduced)
    )

    difference = original_accuracy - reduced_accuracy

    print(
        f"Removed: {feature:30s} "
        f"Accuracy: {reduced_accuracy:.6f} "
        f"Difference: {difference:.6f}"
    )

Removed: lead_source                    Accuracy: 0.642000 Difference: 0.003000
Removed: number_of_courses_viewed       Accuracy: 0.643000 Difference: 0.002000
Removed: interaction_count              Accuracy: 0.601000 Difference: 0.044000


# Q6 — Regularization

In [37]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model_reg = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )

    model_reg.fit(X_train, y_train)

    y_pred = model_reg.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)

    print(f"C={C}: accuracy={accuracy:.3f}")

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
